# SOLA 4237 Assignment 2 - Household Battery Energy Storage System (BESS) Optimisation

### Site: `23W0158` - `NSW Thredbo, Alpine` - `2022` / `2023`

### SOLA4237 Consumer Energy Resources (CER) Operations and Integration

<div style="font-size: 1rem; font-weight: normal; line-height: 1.4;">
Student: <b>Bilal Parkar</b> &nbsp;|&nbsp; zID: <b>z5476566</b><br>
Prepared for: VPP-SPREE single-site optimisation division (retailer client review)
</div>

---

### Purpose and Context
This notebook holds **all analysis and optimisation code** behind the Assignment 2 report. The report must show that VPP-SPREE's single-site optimisation can deliver real, quantified benefit to a household across **flat, time-of-use (ToU) and wholesale pass-through tariffs**, and that it still works when **PV and price forecasts are imperfect**.

| # | Question the report must answer | Evidence built in |
|---|---|---|
| 1 | What are the site, data quality and tariff context? | Parts 1-4 |
| 2 | What battery (size, power, efficiency, limits, degradation) suits this site? | Parts 4, 5, 8 |
| 3 | What is the benchmark (no battery, simple self-consumption)? | Part 6 |
| 4 | What is the best possible operation on each tariff (perfect foresight)? | Part 7 |
| 5 | How good are our PV and price forecasts? | Parts 9, 10 |
| 6 | How much value is lost to uncertainty, and is PV or price the bigger driver? | Parts 11, 12 |
| 7 | Annual bill, self-consumption, savings, payback, VPP fee vs value | Part 13 |

### How this notebook is organised
- **Part 0** setup and constants. **Parts 1-3** load, check and prepare data. **Part 4** characterises the site. **Part 5** is the battery model. **Part 6** benchmarks. **Part 7** perfect-foresight optimiser. **Part 8** sizing and payback. **Parts 9-10** forecasts. **Part 11** 48-hour rolling-horizon scheduler with 5-minute execution. **Part 12** perfect vs uncertain comparison. **Part 13** techno-economics. **Part 14** discussion and recommendations. **Part 15** report figures. **Appendix A** AI disclosure and verification log. **Appendix B** reproducibility.
- Each task follows one pattern: *markdown explanation -> function -> test -> apply to site data -> findings cell*.
- Functions are tested on tiny synthetic samples (`assert`) before they touch site data.
- **Findings cells** (blockquotes) record what the data showed. Anything marked `TODO` is not finished.
- Set `SOLA_DATA_PATH` (or edit `DATA_PATH` in Part 0) and run top to bottom. Set `QUICK_RUN_DAYS` in Part 0 for a fast dry run of the slow parts (11-12).

### Required files
- Assignment CSV (two years of 5-minute data). Column names are mapped in one place (`COL` in Part 0): **check these against your CSV first.**
- AEMO pre-dispatch price forecasts for the site's region (only if you choose them over your own price model; see Part 10).

### Contents and status

| Part | Title | Status | Note |
|---|---|---|---|
| 0 | Setup and configuration | Skeleton | Fill site metadata, tariff windows, cost assumptions |
| 1 | Data loading and integrity | Done | Findings 1.2-1.6 and Part 1 summary written |
| 2 | Data cleaning | Skeleton | Fix only what the Part 1 summary lists |
| 3 | Feature engineering and tariffs | Skeleton | Wholesale retail price build needs sourced network/fee values |
| 4 | Site characterisation | Skeleton | Load vs PV, shiftable energy, topology assumption |
| 5 | Battery model | Skeleton | Choose real product spec; temperature derating needs a source |
| 6 | Benchmarks and billing | Skeleton | No battery, self-consumption |
| 7 | Perfect-foresight optimiser | Skeleton | LP written and tested; run on 3 tariffs |
| 8 | Sizing and payback | Skeleton | Capex assumptions need sources |
| 9 | PV forecasting | Skeleton | Own model vs commercial forecast |
| 10 | Price forecasting | Skeleton | AEMO pre-dispatch or own model |
| 11 | Rolling-horizon scheduler | Skeleton | 48 h hourly plan, 5 min execution |
| 12 | Perfect vs uncertain; PV vs price | Skeleton | Ablation study |
| 13 | Techno-economics and metrics | Skeleton | Master results table |
| 14 | Discussion and recommendations | Skeleton | Fill after results |
| 15 | Report figures | Skeleton | Planned figure list |
| A | AI disclosure and verification log | Skeleton | Fill as you work, not at the end |
| B | Reproducibility | Skeleton | Versions, seed, runtime |

#### Brief requirements tracker

| # | Requirement from the brief | Where | Status |
|---|---|---|---|
| 1 | Default self-consumption mode as benchmark | Part 6 | TODO |
| 2 | AC vs DC bus assumption, losses | Parts 4, 5 | TODO |
| 3 | BESS size, power, SOC limits, temperature limits, degradation cost | Parts 5, 8 | TODO |
| 4 | Optimal strategy for flat, ToU, wholesale | Part 7 | TODO |
| 5 | Wholesale retail price (network, fees, certificates, hedging) | Part 3 | TODO |
| 6 | $25/month VPP fee on wholesale pass-through | Part 13 | TODO |
| 7 | Zero export (curtail) when prices are negative | Part 7 | TODO |
| 8 | PV forecast model trained on Year 1, compared with commercial forecast (MAE, RMSE, nRMSE) | Part 9 | TODO |
| 9 | Price forecast (own model or AEMO pre-dispatch) | Part 10 | TODO |
| 10 | 48 h rolling horizon, hourly replanning, 5-minute execution without knowing current-interval price/solar | Part 11 | TODO |
| 11 | Settle on actual price, BESS flows, load and solar | Part 11 | TODO |
| 12 | Uncertainty vs perfect foresight; PV vs price influence | Part 12 | TODO |
| 13 | Annual bill, self-consumption, savings, payback | Part 13 | TODO |
| 14 | AI disclosure with evidence of checking AI output | Appendix A | TODO |

# Part 0 - Setup and Configuration

### Conventions used throughout (verify in Part 2)
- **Battery power** `batt_kW`: **positive = discharging**, **negative = charging** (same convention as Assignment 1).
- **Grid power** `grid_kW`: **positive = import**, **negative = export**.
- **Accounting identity:** `grid = load - (solar - curtailed) - battery`.
- **Power to energy:** `kWh = kW x dt_h` (`dt_h = 5/60` for raw data, `1` for hourly plans).
- **Prices** are in **$/kWh** inside the code. Wholesale data arrives in $/MWh, so divide by 1000.
- **Time:** timestamps are naive local time. State whether they are NEM time (fixed AEST, UTC+10) in Part 1 and align prices accordingly.

In [1]:
import os, io, warnings, dataclasses
from dataclasses import dataclass, replace
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import linprog
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
RNG_SEED = 42
np.random.seed(RNG_SEED)

In [2]:
# ---------- Paths ----------
DATA_PATH = os.environ.get(
    "SOLA_DATA_PATH",
    "/Users/bilalp/Documents/SOLA4237/CER_Assignments/Assignment_2/SOLA/Data_NSW_Thredbo - Alpine_2022-2023_partial.csv",
)
# ---------- Column map: the ONLY place raw CSV names appear ----------
COL = dict(
    time     = "timestamp",
    load     = "load_kW",
    solar    = "solar_output_kW",                        # measured PV (expected Year 1 only)
    solar_fc = "commercial_forecasted_solar_output_kW",  # commercial forecast (expected Year 2 only)
    irr      = "irradiance_W/m^2",
    temp     = "temperature_C",
    rrp      = "spot_price_AUD/kWh",                     # already $/kWh: do NOT divide by 1000
    flat_imp = "tariff_import_flat_AUD/kWh",
    flat_exp = "tariff_export_flat_AUD/kWh",
    tou_imp  = "tariff_import_tou_AUD/kWh",
    tou_exp  = "tariff_export_tou_AUD/kWh",
)

# ---------- Site metadata (from CER-data-generator screenshot) ----------
SITE_ID      = "NSW_Thredbo"
LOCATION     = "Thredbo, NSW (Alpine)"
NEM_REGION   = "NSW1"
LOAD_PROFILE = "23W0158 (scaled x29.1)"
PV_PR        = 0.85      # performance ratio
PV_GAMMA     = -0.004    # power temperature coefficient, per degC

# ---------- Time ----------
INTERVAL_MIN   = 5
DT_H           = INTERVAL_MIN / 60      # hours per interval: kWh = kW * DT_H
STEPS_PER_HOUR = 60 // INTERVAL_MIN

# Part 1 - Data Loading and Integrity

### Background: why check the timeline before anything else?
Every later calculation assumes one row per 5-minute interval. A missing or duplicated interval silently biases energy totals, and a clock shift (daylight saving) misaligns solar noon and price merges. "No nulls" is not the same as "complete". This dataset also mixes **two years with different columns populated** (measured solar in Year 1, commercial forecast in Year 2), so nulls in those columns are expected, not faults.

## Task 1.1 - Load the telemetry

In [3]:
def load_telemetry(path) -> pd.DataFrame:
    out = pd.read_csv(path, parse_dates=["timestamp"])
    return out.set_index("timestamp").sort_index()

In [4]:
"""Tests for Task 1.1"""
_csv = io.StringIO("timestamp,x\n2024-01-01 00:05:00,2\n2024-01-01 00:00:00,1\n")
_t = load_telemetry(_csv)
assert isinstance(_t.index, pd.DatetimeIndex)   # real datetimes, not strings
assert _t.index.is_monotonic_increasing          # sorted
assert _t["x"].tolist() == [1, 2]                # values travelled with their timestamps
print("Task 1.1 tests passed")

Task 1.1 tests passed


In [5]:
df = load_telemetry(DATA_PATH)
print(df.shape)
df.head()

(210240, 10)


,irradiance_W/m^2,temperature_C,solar_output_kW,commercial_forecasted_solar_output_kW,load_kW,spot_price_AUD/kWh,tariff_import_flat_AUD/kWh,tariff_export_flat_AUD/kWh,tariff_import_tou_AUD/kWh,tariff_export_tou_AUD/kWh
timestamp,,,,,,,,,,
2022-01-01 00:00:00,0.0,15.1,0.0,NaN,0.53,0.12477,0.3,0.05,0.15,0.05
2022-01-01 00:05:00,0.0,15.1,0.0,NaN,0.54,0.12486,0.3,0.05,0.15,0.05
2022-01-01 00:10:00,0.0,15.1,0.0,NaN,0.53,0.12602,0.3,0.05,0.15,0.05
2022-01-01 00:15:00,0.0,15.0,0.0,NaN,0.51,0.12493,0.3,0.05,0.15,0.05
2022-01-01 00:20:00,0.0,15.0,0.0,NaN,0.52,0.11354,0.3,0.05,0.15,0.05


## Task 1.2 - Timeline integrity and the two-year split
`check_timeline(df)` reports rows, expected rows, missing timestamps, duplicates. `split_years(df)` cuts the data at exactly one year after the first timestamp, so a financial-year dataset also splits correctly.

In [6]:
def check_timeline(df : pd.DataFrame, freq: str = "5min") -> dict:
    
    """Check the index is a complete, duplicate-free, regular time series.

    Compares the actual timestamps against an ideal grid from first to last
    timestamp at `freq`. Note: a complete timeline can still hold empty rows,
    so check nulls separately.
    """
    
    full = pd.date_range(df.index.min(), df.index.max(), freq = freq)

    return {
        "rows": len(df),
        "expected": len(full),
        "missing": len(full.difference(df.index)),
        "duplicates": int(df.index.duplicated().sum()),
        "monotonic": bool(df.index.is_monotonic_increasing),
        "start": df.index.min(),
        "end": df.index.max()
    }

In [7]:
"""Tests for Task 1.2a"""
# one missing interval (00:10)
_idx = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 00:05", "2024-01-01 00:15"])
_r = check_timeline(pd.DataFrame({"a": [1, 2, 3]}, index=_idx))
assert _r["rows"] == 3 and _r["expected"] == 4 and _r["missing"] == 1 and _r["duplicates"] == 0

# the cancelling case: one gap AND one duplicate -> row count looks perfect but data isn't
_idx = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 00:05", "2024-01-01 00:05", "2024-01-01 00:15"])
_r = check_timeline(pd.DataFrame({"a": [1, 2, 3, 4]}, index=_idx))
assert _r["rows"] == _r["expected"] == 4          # counts match...
assert _r["missing"] == 1 and _r["duplicates"] == 1   # ...but it's still broken
print("Task 1.2a tests passed")

Task 1.2a tests passed


In [8]:
def split_years(df:pd.DataFrame):

    """Split into (year1, year2) at exactly one year after the first timestamp.

    Uses DateOffset rather than a fixed 365 days so leap years split correctly.
    """
    
    cut = df.index.min() + pd.DateOffset(years=1)
    return df[df.index < cut], df[df.index >= cut]

In [9]:
"""Tests for Task 1.2b"""
_idx2 = pd.date_range("2023-01-01", "2024-12-31 23:55", freq="5min")
_y1, _y2 = split_years(pd.DataFrame({"a": 1}, index=_idx2))
assert _y1.index.max() < _y2.index.min()                      # no overlap
assert len(_y1) + len(_y2) == len(_idx2)                       # nothing lost
assert len(_y1) == 365 * 288 and len(_y2) == 366 * 288
print("Task 1.2b tests passed")

Task 1.2b tests passed


In [10]:
timeline = check_timeline(df)
print(pd.Series(timeline))
df1, df2 = split_years(df)
print("Year 1 rows:", len(df1), "| Year 2 rows:", len(df2))
pd.concat({"year1_nulls": df1.isna().sum(), "year2_nulls": df2.isna().sum()}, axis=1)

rows                       210240
expected                   210240
missing                         0
duplicates                      0
monotonic                    True
start         2022-01-01 00:00:00
end           2023-12-31 23:55:00
dtype: object
Year 1 rows: 105120 | Year 2 rows: 105120


,year1_nulls,year2_nulls
irradiance_W/m^2,184,866
temperature_C,184,866
solar_output_kW,184,105120
commercial_forecasted_solar_output_kW,105120,0
load_kW,184,866
spot_price_AUD/kWh,0,105120
tariff_import_flat_AUD/kWh,0,0
tariff_export_flat_AUD/kWh,0,0
tariff_import_tou_AUD/kWh,0,0
tariff_export_tou_AUD/kWh,0,0


In [11]:
sensor_cols = [COL["irr"], COL["temp"], COL["load"]]

# Are the sensor gaps the same rows? (True = yes)
blank = df[sensor_cols].isna()
print("Same rows across sensor columns:", (blank.all(axis=1) == blank.any(axis=1)).all())

# Group consecutive blank rows into gaps: start, end, length
is_gap = blank.any(axis=1)
gap_id = (is_gap != is_gap.shift()).cumsum()
gaps = (df[is_gap].groupby(gap_id[is_gap])
        .apply(lambda g: pd.Series({"start": g.index.min(), "end": g.index.max(), "intervals": len(g)})))
gaps["hours"] = gaps["intervals"] * DT_H
print(gaps.sort_values("intervals", ascending=False).head(15))
print("Number of gaps:", len(gaps))


Same rows across sensor columns: True
                 start                 end  intervals      hours
32 2023-07-01 22:20:00 2023-07-02 22:15:00        288  24.000000
12 2022-12-27 14:10:00 2022-12-27 19:50:00         69   5.750000
36 2023-08-12 07:10:00 2023-08-12 12:40:00         67   5.583333
30 2023-06-23 06:45:00 2023-06-23 12:15:00         67   5.583333
18 2023-03-15 13:50:00 2023-03-15 18:40:00         59   4.916667
42 2023-12-17 22:45:00 2023-12-18 03:25:00         57   4.750000
34 2023-07-23 22:15:00 2023-07-24 02:40:00         54   4.500000
28 2023-06-17 06:45:00 2023-06-17 10:45:00         49   4.083333
2  2022-03-13 10:50:00 2022-03-13 14:45:00         48   4.000000
38 2023-10-10 04:05:00 2023-10-10 08:00:00         48   4.000000
24 2023-04-29 07:10:00 2023-04-29 11:05:00         48   4.000000
22 2023-03-27 05:15:00 2023-03-27 07:10:00         24   2.000000
10 2022-09-13 15:55:00 2022-09-13 17:40:00         22   1.833333
44 2023-12-29 06:20:00 2023-12-29 08:05:00         2

> **Finding 1.2 - Completeness.** The timeline is complete: 210,240 of 210,240 expected 5-minute intervals, with no missing timestamps and no duplicates. Because there are no gaps or repeats around October and April, the timestamps are on a fixed offset with no daylight-saving jumps, consistent with NEM time (AEST); this is checked against solar noon in Part 4.
>
> Some columns are empty by design. Measured PV exists only in Year 1 and the commercial forecast only in Year 2, so the two are exact complements. There are no Year 2 spot prices, so actual 2023 NSW1 prices must be sourced from AEMO to settle the Year 2 wholesale bill.
>
> A complete timeline is not complete data. 22 data outages leave whole rows blank: the same rows in irradiance, temperature, load and PV (184 intervals in Year 1, 866 in Year 2, about 0.5% of the data). They range up to 24 hours (1–2 July 2023), and several cover the morning solar ramp. Handling is deferred to Part 2.

## Task 1.3 - Plausibility of each column

### Background: present is not the same as possible
Task 1.2 showed the data exists; this task checks it is physically possible. Each column has a hard physical bound: irradiance and load cannot be negative, PV is zero at night and capped by system size, and NEM spot prices sit between the market floor (-$1,000/MWh) and the market price cap. A value outside its bound means a unit or sensor problem that every later calculation would inherit. Year 1 and Year 2 are summarised separately so measured PV (Year 1) can be compared like-for-like with the commercial forecast (Year 2).

In [12]:
# Ranges per year (Year 2 PV is the forecast, so compare like with like)
desc = pd.concat({"Y1": df1.describe().T, "Y2": df2.describe().T})
desc[["min", "50%", "max", "mean"]].round(3)

min     50%      max     mean
Y1 irradiance_W/m^2                        0.00  13.000  1137.00  168.650
   temperature_C                          -8.20   6.400    24.30    6.816
   solar_output_kW                         0.00   0.100    10.31    1.263
   commercial_forecasted_solar_output_kW    NaN     NaN      NaN      NaN
   load_kW                                 0.17   0.570     5.87    1.212
   spot_price_AUD/kWh                     -1.00   0.134    15.50    0.183
   tariff_import_flat_AUD/kWh              0.30   0.300     0.30    0.300
   tariff_export_flat_AUD/kWh              0.05   0.050     0.05    0.050
   tariff_import_tou_AUD/kWh               0.15   0.250     0.40    0.242
   tariff_export_tou_AUD/kWh               0.05   0.050     0.05    0.050
Y2 irradiance_W/m^2                        0.00  16.000  1104.00  191.172
   temperature_C                         -10.10   7.400    26.20    7.805
   solar_output_kW                          NaN     NaN      NaN      NaN
   commercial_forecasted_solar_output_kW   0.00   0.120     8.25    1.458
   load_kW                                 0.17   0.570     5.87    1.215
   spot_price_AUD/kWh                       NaN     NaN      NaN      NaN
   tariff_import_flat_AUD/kWh              0.30   0.300     0.30    0.300
   tariff_export_flat_AUD/kWh              0.05   0.050     0.05    0.050
   tariff_import_tou_AUD/kWh               0.15   0.250     0.40    0.242
   tariff_export_tou_AUD/kWh               0.05   0.050     0.05    0.050

> **Finding 1.3 - Ranges.** Every column is physically plausible. Irradiance ranges from 0 to 1,137 W/m² (Year 1) and 0 to 1,104 W/m² (Year 2), consistent with clear-sky noon at about 1,000 W/m² plus brief cloud-edge enhancement and the site's high altitude. Temperature runs from −10.1 °C to 26.2 °C with a mean of about 7 °C, as expected for an alpine site, so cold-weather battery derating will matter (Part 5). Load is always positive (0.17–5.87 kW, mean 1.21 kW, about 29 kWh/day), so there is never zero standby load.
>
> The spot price reaches exactly −1.00 and 15.50 AUD/kWh, the NEM market floor (−1,000 AUD/MWh) and cap (15,500 AUD/MWh). This confirms the column is already in AUD/kWh and must not be divided by 1,000. Its mean of 183 AUD/MWh reflects the 2022 energy crisis.
>
> Measured PV peaks at 10.3 kW near 1,100 W/m², implying a system of roughly 10 kW (refined in Part 4). The Year 2 commercial forecast has a higher mean than Year 1 measured PV (1.46 vs 1.26 kW), in line with Year 2's sunnier irradiance, but a lower peak (8.25 vs 10.31 kW). Year 2 irradiance peaked only 3% lower than Year 1, so weather does not explain a 20% lower PV peak; this suggests the forecast is smoothed and misses short peaks, which is examined in Part 9.

## Task 1.4 - Tariff structure

### Background: read the tariff from the data, not from assumptions
The battery's value on a ToU tariff depends entirely on *when* each rate applies, so the windows must come from the data. `tou_schedule` builds an hour-by-hour table of the minimum and maximum rate, split by weekday and weekend. Taking min and max over the whole dataset tests three things at once: if they are equal for every hour, the boundaries fall on whole hours, the schedule does not change with season, and it applies identically to the days in each column.

In [13]:
def tou_schedule(rate: pd.Series) -> pd.DataFrame:
    """Hour-of-day x weekday/weekend table of the min and max rate in each hour.

    min == max in every cell means whole-hour boundaries and no seasonal change.
    min != max flags a mid-hour boundary or a rate that varies across the year.
    """
    t = pd.DataFrame({"hour": rate.index.hour,
                      "weekend": rate.index.dayofweek >= 5,
                      "rate": rate.values})
    return t.pivot_table(index="hour", columns="weekend", values="rate", aggfunc=["min", "max"])

In [14]:
"""Tests for Task 1.4"""
# Saturday 6 Jan and Monday 8 Jan 2024; peak 19:00-23:00, otherwise off-peak
_i = pd.date_range("2024-01-06", periods=288, freq="5min").append(pd.date_range("2024-01-08", periods=288, freq="5min"))
_r = pd.Series(np.where((_i.hour >= 19) & (_i.hour < 23), 0.40, 0.15), index=_i)
_s = tou_schedule(_r)
assert _s[("min", False)].loc[19] == 0.40 and _s[("min", True)].loc[19] == 0.40     # peak on both day types
assert _s[("max", False)].loc[3] == 0.15                                             # off-peak overnight
assert (_s["min"] == _s["max"]).all().all()                                          # whole-hour windows

# A boundary at 16:30 must show up as min != max in hour 16
_r2 = pd.Series(np.where(_i.hour + _i.minute / 60 >= 16.5, 0.40, 0.15), index=_i)
_s2 = tou_schedule(_r2)
assert _s2[("min", False)].loc[16] == 0.15 and _s2[("max", False)].loc[16] == 0.40
print("Task 1.4 tests passed")

Task 1.4 tests passed


In [15]:
for c in ["flat_imp", "flat_exp", "tou_imp", "tou_exp"]:
    print(f"{c:9s}", sorted(df[COL[c]].dropna().unique()))
tou_schedule(df[COL["tou_imp"]])

flat_imp  [np.float64(0.3)]
flat_exp  [np.float64(0.05)]
tou_imp   [np.float64(0.15), np.float64(0.25), np.float64(0.4)]
tou_exp   [np.float64(0.05)]


min         max      
weekend False True  False True 
hour                           
0        0.15  0.15  0.15  0.15
1        0.15  0.15  0.15  0.15
2        0.15  0.15  0.15  0.15
3        0.15  0.15  0.15  0.15
4        0.15  0.15  0.15  0.15
5        0.15  0.15  0.15  0.15
6        0.15  0.15  0.15  0.15
7        0.25  0.25  0.25  0.25
8        0.25  0.25  0.25  0.25
9        0.25  0.25  0.25  0.25
10       0.25  0.25  0.25  0.25
11       0.25  0.25  0.25  0.25
12       0.25  0.25  0.25  0.25
13       0.25  0.25  0.25  0.25
14       0.25  0.25  0.25  0.25
15       0.25  0.25  0.25  0.25
16       0.25  0.25  0.25  0.25
17       0.25  0.25  0.25  0.25
18       0.25  0.25  0.25  0.25
19       0.40  0.40  0.40  0.40
20       0.40  0.40  0.40  0.40
21       0.40  0.40  0.40  0.40
22       0.40  0.40  0.40  0.40
23       0.15  0.15  0.15  0.15

In [16]:
# What is one kWh moved by the battery worth on each tariff? (before losses)
h = df.index.hour
n_days = df.index.normalize().nunique()
peak_kwh_day = df.loc[(h >= 19) & (h < 23), COL["load"]].sum() * DT_H / n_days
print(f"Average load in the 19:00-23:00 peak window: {peak_kwh_day:.1f} kWh/day")

exp_rate = 0.05
pd.DataFrame({
    "store own solar, use later (AUD/kWh)":   {"Flat": 0.30 - exp_rate, "ToU into off-peak": 0.15 - exp_rate,
                                                "ToU into shoulder": 0.25 - exp_rate, "ToU into peak": 0.40 - exp_rate},
    "charge from grid off-peak, use later":  {"Flat": 0.0, "ToU into off-peak": 0.0,
                                                "ToU into shoulder": 0.25 - 0.15, "ToU into peak": 0.40 - 0.15},
}).round(2)

Average load in the 19:00-23:00 peak window: 2.6 kWh/day


,"store own solar, use later (AUD/kWh)","charge from grid off-peak, use later"
Flat,0.25,0.00
ToU into off-peak,0.10,0.00
ToU into shoulder,0.20,0.10
ToU into peak,0.35,0.25


> **Finding 1.4 - Tariffs.** The tariff columns match the supplied metadata (flat 0.30/0.05 AUD/kWh; ToU 0.15/0.25/0.40 AUD/kWh import, 0.05 export). The metadata gives the rates but not their timing; the data shows:
>
> | Window | Hours | Import (AUD/kWh) |
> |---|---|---|
> | Off-peak | 23:00 - 07:00| 0.15 |
> | Shoulder | 07:00 - 19:00 | 0.25 |
> | Peak | 19:00 - 23:00 | 0.40 |
>
> The schedule is identical on weekdays and weekends with no seasonal change. Export earns 0.05 AUD/kWh under every tariff.
>
> **So what for the battery.** Export is worth less than every import rate, so on every tariff a stored kWh of solar beats an exported one. On the **flat** tariff that is the battery's only value (0.25 AUD/kWh shifted, before losses); there is no price spread to arbitrage. On **ToU** the value depends on *when* the stored energy is used: 0.35 AUD/kWh into the peak, 0.20 into the shoulder, but only 0.10 overnight. ToU also opens **grid arbitrage**: charging off-peak and discharging in the peak is worth 0.25 AUD/kWh before losses, which matters in winter when there is little surplus solar. The peak window is only 4 hours and carries about 2.6 kWh/day of load, so a battery sized for the peak alone would be small; the sizing question (Part 4) is how much of the evening and overnight load is worth covering.


## Task 1.5 - Is the load independent across years?

### Background: check whether two years are really two years
The Year 1 and Year 2 load summaries in Task 1.3 were almost identical. `compare_years` lines the two years up interval by interval (by calendar position) and measures how often the values are identical, how correlated they are, and how many gaps in each year could be filled from the other. Irradiance is run through the same check as a control: real weather should not repeat.

In [17]:
def compare_years(y1: pd.Series, y2: pd.Series) -> dict:
    """Compare two equal-length years aligned by calendar position.

    Positional alignment needs equal-length years (true for 2022/2023, both 365 days);
    a leap year would shift every interval after 29 Feb, so the function refuses.
    """
    a, b = np.asarray(y1, float), np.asarray(y2, float)
    if len(a) != len(b):
        raise ValueError(f"Years differ in length ({len(a)} vs {len(b)}): positional alignment invalid")
    both = ~np.isnan(a) & ~np.isnan(b)
    return dict(
        identical_frac=float(np.isclose(a[both], b[both]).mean()),
        corr=float(np.corrcoef(a[both], b[both])[0, 1]),
        y1_gaps=int(np.isnan(a).sum()), y1_fillable_from_y2=int((np.isnan(a) & ~np.isnan(b)).sum()),
        y2_gaps=int(np.isnan(b).sum()), y2_fillable_from_y1=int((np.isnan(b) & ~np.isnan(a)).sum()),
        missing_both=int((np.isnan(a) & np.isnan(b)).sum()),
    )

In [18]:
"""Tests for Task 1.5"""
_c = compare_years([1, 2, np.nan, 4], [1, np.nan, 3, 4])
assert _c["identical_frac"] == 1.0                                   # compared only where both exist
assert _c["y1_fillable_from_y2"] == 1 and _c["y2_fillable_from_y1"] == 1 and _c["missing_both"] == 0

_c = compare_years([1, 2, 3, 4], [1, 2, 3, 9])
assert _c["identical_frac"] == 0.75

try:
    compare_years([1, 2, 3], [1, 2])
    raise AssertionError("should have refused unequal lengths")
except ValueError:
    pass
print("Task 1.5 tests passed")

Task 1.5 tests passed


In [19]:
pd.DataFrame({
    "load": compare_years(df1[COL["load"]], df2[COL["load"]]),
    "irradiance (control)": compare_years(df1[COL["irr"]], df2[COL["irr"]]),
}).round(4)

,load,irradiance (control)
identical_frac,1.0,0.4570
corr,1.0,0.8482
y1_gaps,184.0,184.0000
y1_fillable_from_y2,184.0,184.0000
y2_gaps,866.0,866.0000
y2_fillable_from_y1,866.0,866.0000
missing_both,0.0,0.0000


In [20]:
# Weekday labels: 1 Jan 2022 was a Saturday, 1 Jan 2023 a Sunday, so a copied year shifts weekdays by one day
print(df1.index[0].day_name(), "|", df2.index[0].day_name())
(df1[COL["load"]].groupby(df1.index.dayofweek).mean() * 24).round(1).rename("Y1 load kWh/day by weekday (0 = Mon)")

Saturday | Sunday


timestamp
0    28.8
1    28.8
2    28.2
3    29.1
4    29.6
5    29.8
6    29.3
Name: Y1 load kWh/day by weekday (0 = Mon), dtype: float64

> **Finding 1.5 - Load is repeated.** Year 2 load equals Year 1 load at every interval where both exist (identical fraction and correlation both 1.0), while irradiance does not (46% identical, r = 0.85). So this is a property of the load data, not a bug in the check. Implications:
> 1. Year 2 is not independent load data. Any difference in battery value between the years comes only from PV, weather and prices, which makes the years a clean comparison of those drivers.
> 2. The copy is by calendar date, so Year 2 weekday labels are shifted one day against the load behaviour. Year 1 load barely varies by weekday (28.2-29.8 kWh/day), so this has no practical effect, but day-of-week should not be used as a feature.
> 3. No interval is missing in both years, so all 1,050 load gaps can be filled exactly from the other year (Part 2).

## Task 1.6 - Spurious data

### Background: in range is not the same as consistent
Task 1.3 checked that every value is *possible* on its own. A value can still be wrong if it contradicts how the sensor behaves over time, or what another sensor says at the same moment. `quality_report` runs three kinds of check:

| Kind | Question | Checks |
|---|---|---|
| Range | Is each value possible on its own? | Negative or zero load, negative irradiance or PV, prices outside the NEM floor and cap. These repeat Task 1.3 as a formal record |
| Behaviour over time | Does the sensor behave like a real sensor? | Longest constant run (stuck meter or sensor), largest 5-minute temperature jump |
| Cross-sensor | Do two sensors measuring the same sunlight agree? | PV when irradiance is zero, no PV under strong sun (outage, snow or inverter trip, plausible at an alpine site), PV above system capacity, daily PV-to-irradiance ratio |

Cross-sensor checks are the strongest evidence, because one sensor checks the other. Each year is checked against its own PV column: measured PV in Year 1, the commercial forecast in Year 2. `quality_report` is adapted from my Assignment 1 data-quality check, with the PV column and system capacity as arguments.

In [ ]:
def longest_constant_run(s: pd.Series) -> int:
    """Length (in intervals) of the longest run of identical consecutive values."""
    s = s.dropna()
    return int(s.groupby((s != s.shift()).cumsum()).size().max()) if len(s) else 0


def quality_report(df: pd.DataFrame, pv_col: str, cap_kw: float, col: dict = COL) -> pd.DataFrame:
    """Physical-plausibility checks in one table. Columns: check, value, note.

    pv_col: which PV column to check (measured in Year 1, commercial forecast in Year 2).
    cap_kw: estimated PV system size, used for the above-capacity check.
    """
    irr, load = df[col["irr"]], df[col["load"]]
    rows = [
        ("Negative load intervals", int((load < 0).sum()), "Should be 0 (no export-side meter in load)"),
        ("Zero-load intervals", int((load == 0).sum()), "A real home always has standby load"),
        ("Longest constant load run (intervals)", longest_constant_run(load), "Stuck meter if very long"),
        ("Longest constant temperature run (intervals)", longest_constant_run(df[col["temp"]]), "Stuck sensor if very long"),
        ("Max 5-min temperature jump (degC)", float(df[col["temp"]].diff().abs().max()), "Implausible if large"),
        ("Negative irradiance intervals", int((irr < 0).sum()), "Should be 0"),
    ]
    s = df[pv_col].dropna()
    if len(s):
        i = irr.reindex(s.index)
        rows += [
            ("PV rows checked", len(s), "Proves the check actually ran"),
            ("Solar > 0.01 kW at zero irradiance", int(((i == 0) & (s > 0.01)).sum()), "Should be 0"),
            ("Negative solar intervals", int((s < 0).sum()), "Should be 0"),
            ("Daytime (irr>100) intervals with solar <= 0.01 kW",
             int(((i > 100) & (s <= 0.01)).sum()), "Outage, snow or fault"),
            (f"Solar above 1.1 x {cap_kw} kW", int((s > 1.1 * cap_kw).sum()), "Above system capacity"),
        ]
    p = df[col["rrp"]].dropna()
    if len(p):
        rows += [("Prices outside [-1.00, 15.50] $/kWh", int(((p < -1.00) | (p > 15.50)).sum()),
                  "Outside 2022 NEM floor/cap: check units"),
                 ("Negative-price intervals", int((p < 0).sum()), "Drives the curtailment logic")]
    return pd.DataFrame(rows, columns=["check", "value", "note"])

In [ ]:
"""Tests for Task 1.6"""
assert longest_constant_run(pd.Series([1, 1, 1, 2, 2])) == 3
assert longest_constant_run(pd.Series([np.nan, np.nan])) == 0            # empty column must not crash

_idx = pd.date_range("2022-01-01 12:00", periods=4, freq="5min")
_q = pd.DataFrame({
    COL["irr"]:      [0, 0, 500, 500],
    COL["load"]:     [0.0, 0.3, 0.3, 0.3],
    COL["temp"]:     [5, 5, 5, 6],
    COL["solar"]:    [0.0, 0.5, 0.0, 12.0],        # PV in the dark, no PV in sun, above 1.1 x 10 kW
    COL["solar_fc"]: np.nan,
    COL["rrp"]:      [0.10, -0.05, 16.0, 0.10],    # one negative price, one above the cap
}, index=_idx)
_r = quality_report(_q, COL["solar"], cap_kw=10).set_index("check")["value"]
assert _r["Zero-load intervals"] == 1
assert _r["Solar > 0.01 kW at zero irradiance"] == 1
assert _r["Daytime (irr>100) intervals with solar <= 0.01 kW"] == 1
assert _r["Solar above 1.1 x 10 kW"] == 1
assert _r["Prices outside [-1.00, 15.50] $/kWh"] == 1 and _r["Negative-price intervals"] == 1

# Year 2 shape: measured PV empty, forecast present -> the forecast is checked, not silently skipped
_y2 = _q.assign(**{COL["solar"]: np.nan, COL["solar_fc"]: [0.0, 0.5, 0.0, 12.0]})
_r2 = quality_report(_y2, COL["solar_fc"], cap_kw=10).set_index("check")["value"]
assert _r2["PV rows checked"] == 4 and _r2["Solar above 1.1 x 10 kW"] == 1
# an empty PV column skips the PV checks instead of reporting false zeros
assert "PV rows checked" not in quality_report(_y2, COL["solar"], cap_kw=10)["check"].values
print("Task 1.6 tests passed")

In [22]:
pd.concat({"Y1": quality_report(df1, COL["solar"], 10).set_index("check")["value"],
           "Y2": quality_report(df2, COL["solar_fc"], 10).set_index("check")["value"]}, axis=1)

,Y1,Y2
check,,
Negative load intervals,0.0,0.0
Zero-load intervals,0.0,0.0
Longest constant load run (intervals),14.0,14.0
Longest constant temperature run (intervals),73.0,81.0
Max 5-min temperature jump (degC),0.8,0.5
Negative irradiance intervals,0.0,0.0
PV rows checked,104936.0,105120.0
Solar > 0.01 kW at zero irradiance,0.0,0.0
Negative solar intervals,0.0,0.0


In [ ]:
def daily_pv_irr_ratio(pv: pd.Series, irr: pd.Series) -> pd.Series:
    """Daily PV energy divided by daily irradiation (kWh per kWh/m^2, i.e. the array's effective kW).

    Uses complete days only: an irradiance outage with no matching PV gap divides
    by too little and inflates the ratio (to infinity if only night-time irradiance is left).
    """
    complete = pd.concat([pv, irr], axis=1).notna().all(axis=1).groupby(pv.index.normalize()).all()
    irr_day = irr.resample("D").sum() / 1000
    ratio = pv.resample("D").sum() / irr_day
    return ratio[complete & (irr_day > 0)]

In [ ]:
"""Tests for Task 1.6 (ratio)"""
_i = pd.date_range("2024-01-01", periods=2 * 288, freq="5min")
_irr = pd.Series(np.where((_i.hour >= 6) & (_i.hour < 18), 800.0, 0.0), index=_i)
_pv = _irr / 1000 * 7.5                                     # a 7.5 kW-effective array
_irr_gap = _irr.copy(); _irr_gap.loc["2024-01-02 08:00":"2024-01-02 12:00"] = np.nan
_r = daily_pv_irr_ratio(_pv, _irr_gap)
assert np.isclose(_r.loc["2024-01-01"], 7.5)                # complete day: exact
assert pd.Timestamp("2024-01-02") not in _r.index           # day with a gap: excluded, not inflated
print("Task 1.6 ratio tests passed")

In [ ]:
ratio = pd.DataFrame({
    "Y1 measured":   daily_pv_irr_ratio(df1[COL["solar"]],    df1[COL["irr"]]),
    "Y2 commercial": daily_pv_irr_ratio(df2[COL["solar_fc"]], df2[COL["irr"]]),
})
summary = ratio.describe().loc[["count", "min", "50%", "max"]].round(2)
summary.loc["min day"] = ratio.idxmin().dt.date
summary.loc["max day"] = ratio.idxmax().dt.date
summary

In [ ]:
def longest_run_details(s: pd.Series) -> dict:
    """Where the longest constant run is: start, end, length and value (to judge stuck sensor vs calm night)."""
    s = s.dropna()
    groups = (s != s.shift()).cumsum()
    run = s[groups == s.groupby(groups).size().idxmax()]
    return dict(intervals=len(run), hours=round(len(run) * DT_H, 2),
                start=run.index[0], end=run.index[-1], value_C=run.iloc[0])

pd.DataFrame({"Y1": longest_run_details(df1[COL["temp"]]),
              "Y2": longest_run_details(df2[COL["temp"]])})

In [ ]:
# Is 5-minute irradiance real 5-minute data? If it was interpolated from hourly values,
# the 5-minute steps inside each hour are all the same size (+/-1 W/m^2 for rounding).
irr1 = df1[COL["irr"]]
step = irr1.diff()
inside_hour = (irr1.index.minute != 0) & step.notna()
spread = step[inside_hour].groupby(irr1.index[inside_hour].floor("h")).agg(lambda x: x.max() - x.min())
daylight = irr1.resample("h").mean().reindex(spread.index) > 50
print(f"Daylight hours whose 5-min irradiance steps are all equal: {(spread[daylight] <= 1).mean():.1%}")

> **Finding 1.6 - No spurious data.** Every check passes in both years (104,936 measured PV rows in Year 1 and 105,120 commercial-forecast rows in Year 2): no PV in the dark, no PV dropout under strong sun, nothing above the ~10 kW system capacity, no zero or negative load, and every price within the NEM floor and cap. The 3,114 negative-price intervals in Year 1 (about 3% of the year) are real market events, not errors; they are when PV can be curtailed (Part 7).
>
> **Temperature.** The longest constant runs (73 intervals, about 6 h, in Year 1; 81, about 6.75 h, in Year 2) are both overnight in winter: from the evening of 24 Aug 2022 at -0.8 °C, and the early morning of 5 Jul 2023 at 4.2 °C. Temperature is recorded to 0.1 °C and never moves more than 0.8 °C in 5 minutes, so a flat reading through a calm alpine night is expected. This is not a stuck sensor, so the data is kept.
>
> **PV vs irradiance.** On complete days the daily PV-to-irradiance ratio is very stable: 6.95 to 8.09 (median 7.57) in Year 1 and 7.29 to 8.12 (median 7.72) in Year 2. There are no snow-covered, soiled or faulted days to remove. The ratio is lowest in summer and highest in winter, as expected from the negative temperature coefficient (cold panels are more efficient). It must be computed on complete days only: on days with an irradiance outage it is inflated, because the PV columns have no matching gap. The ratio (~7.6 kW per kW/m²) is also a first estimate of effective array size for Part 4.
>
> **Irradiance resolution.** In every daylight hour, the 5-minute irradiance rises in equal steps, so it is **hourly data interpolated to 5 minutes**, while measured PV is true 5-minute data. Cloud transients therefore appear in PV but not in irradiance. A weather-based PV model (Part 9) cannot predict sub-hourly dips, which sets a floor under its 5-minute error.

## Part 1 summary - what we are working with

Part 1 diagnosed the data without changing it. Part 2 fixes only what this list says needs fixing.

| # | Issue | Found in | What happens next |
|---|---|---|---|
| 1 | 22 sensor outages blank irradiance, temperature, load and measured PV together (184 rows in Y1, 866 in Y2; up to 24 h) | 1.2 | Part 2: fill or flag, column by column |
| 2 | Measured PV only in Y1, commercial forecast only in Y2 | 1.2 | By design. Y2 "actual" PV = commercial forecast (per the brief) |
| 3 | No Y2 spot prices | 1.2 | Part 3: source 2023 NSW1 prices from AEMO |
| 4 | Y2 load is an exact copy of Y1, with weekdays shifted one day | 1.5 | Part 2: fill load gaps from the other year; no day-of-week features |
| 5 | Prices already in $/kWh; hit the floor and cap; 3,114 negative intervals | 1.3, 1.6 | Keep as-is; curtail PV at negative prices (Part 7) |
| 6 | Irradiance is hourly data interpolated to 5 min | 1.6 | Caveat for the PV model (Part 9) |
| 7 | No spurious data; temperature runs are calm winter nights | 1.6 | Nothing to remove |
| 8 | No daylight-saving jumps (likely AEST) | 1.2 | Confirm with solar noon (Part 4) |